# Лабораторна робота 10. Завдання 2. Коні / люди: тестові зображення, 300×300, аугментація, тонке налаштування

Виконав: Бабин Богдан Олегович, група 443-4. Основа — ноутбук Завдання 1 (InceptionV3 до шару mixed7 + Flatten → Dense 1024 → Dropout 0.2 → sigmoid,
RMSprop 1e-4, колбек зупинки на 99,9%). Чотири експерименти в одному ноутбуці, щоб результати було легко порівняти:

| | експеримент | пункт завдання |
|---|---|---|
| A | 150×150, без аугментації (як у Завданні 1) + розпізнавання нових фото | 2 |
| B | оригінальна роздільна здатність **300×300** | 2 |
| C | 150×150 + **Data Augmentation** | 3 |
| D | модель A + **Fine Tuning**: розморожуємо ще один згортковий шар | 4 |

**Перед запуском:** Среда выполнения → T4 GPU → «Выполнить все».

In [ ]:
# --- Сумісність (додано): завдання написане під Keras 2 (layer.output_shape, w.shape.num_elements(), формат summary).
# У сучасному Colab TensorFlow ≥ 2.16 з Keras 3, де цих атрибутів немає, тому вмикаємо Keras 2 (пакет tf_keras).
import os, importlib.util, subprocess, sys
os.environ['TF_USE_LEGACY_KERAS'] = '1'
if importlib.util.find_spec('tf_keras') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tf_keras'])
# ---

import zipfile, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import img_to_array, load_img
from tensorflow.keras.applications.inception_v3 import InceptionV3
tf.keras.utils.set_random_seed(42)
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
!wget -q -P /content/ https://storage.googleapis.com/learning-datasets/horse-or-human.zip
!wget -q -P /content/ https://storage.googleapis.com/learning-datasets/validation-horse-or-human.zip
!wget -q https://storage.googleapis.com/tensorflow/keras-applications/inception_v3/inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5 -O /tmp/inception_v3_weights.h5
zipfile.ZipFile('./horse-or-human.zip').extractall('/tmp/training')
zipfile.ZipFile('./validation-horse-or-human.zip').extractall('/tmp/validation')
train_dir, validation_dir = '/tmp/training', '/tmp/validation'
for d in (train_dir, validation_dir):
    print(d, {c: len(os.listdir(os.path.join(d, c))) for c in sorted(os.listdir(d))})

## Тестові зображення
По 7 фото коней і людей з інтернету (фотобанк pixel-shot.com), яких немає в навчальному і перевірочному наборах,
та 3 «обманні»: дівчина верхи на коні, жінка поруч з конем, жінка в костюмі для верхової їзди (файли `_h`).
Особливість набору коні-люди: усі навчальні фото — **згенеровані 3D-моделі** на однотонному фоні, а тестові — справжні фотографії.

In [ ]:
import re, io, requests
from PIL import Image
UA = {'User-Agent': 'Mozilla/5.0 (student lab project; ChNU)'}
def fetch_images(items, folder):
    '''Завантажує зображення: items = [(ім'я_файлу, адреса сторінки pixel-shot або пряма адреса зображення)]'''
    os.makedirs(folder, exist_ok=True); ok = []
    for name, page in items:
        try:
            if 'pixel-shot.com/en/showcase' in page:
                html = requests.get(page, headers=UA, timeout=20).text
                url = re.search(r'https://img\.pixel-shot\.com/[^"\'\s]+?showcase\.(?:webp|jpg|jpeg|png)', html).group(0)
            else:
                url = page
            r = requests.get(url, headers=UA, timeout=30); r.raise_for_status()
            Image.open(io.BytesIO(r.content)).convert('RGB').save(os.path.join(folder, name + '.jpg'), quality=92)
            ok.append(name)
        except Exception as e:
            print('не вдалося завантажити', name, '-', type(e).__name__)
    print(f'{folder}: завантажено {len(ok)} з {len(items)}')
    return ok

def show_folder(folder, title, cols=7):
    fs = sorted(f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    rows = max(1, (len(fs) + cols - 1) // cols)
    plt.figure(figsize=(cols * 2.3, rows * 2.5))
    for i, f in enumerate(fs):
        plt.subplot(rows, cols, i + 1); plt.imshow(Image.open(os.path.join(folder, f))); plt.title(f, fontsize=8); plt.axis('off')
    plt.suptitle(title); plt.tight_layout(); plt.show()

HH_TEST = [
 [
  "horse_1",
  "https://pixel-shot.com/en/showcase/i-2411418-0/beautiful-horse-in-the-field"
 ],
 [
  "horse_2",
  "https://pixel-shot.com/en/showcase/i-1677548-0/portrait-of-beautiful-brown-horse--outdoors"
 ],
 [
  "horse_3",
  "https://pixel-shot.com/en/showcase/i-1333675-0/beautiful-brown-horse-in-pasture"
 ],
 [
  "horse_4",
  "https://pixel-shot.com/en/showcase/i-1700059-0/pair-of-beautiful-horses--outdoors"
 ],
 [
  "horse_5",
  "https://pixel-shot.com/en/showcase/i-998370-0/portrait-of-purebred-horse-on-nature-background"
 ],
 [
  "horse_6",
  "https://pixel-shot.com/en/showcase/i-1677822-0/beautiful-white-horse-grazing-on-meadow"
 ],
 [
  "horse_7",
  "https://pixel-shot.com/en/showcase/i-2471915-0/beautiful-horse-standing-in-field"
 ],
 [
  "human_1",
  "https://pixel-shot.com/en/showcase/i-786456-0/male-worker-isolated-on-white"
 ],
 [
  "human_2",
  "https://pixel-shot.com/en/showcase/i-558837-0/young-businessman-isolated-on-white"
 ],
 [
  "human_3",
  "https://pixel-shot.com/en/showcase/i-5905011-0/portrait-of-young-man-on-white-background"
 ],
 [
  "human_4",
  "https://pixel-shot.com/en/showcase/i-3099661-0/handsome-doctor-standing-on-white-background"
 ],
 [
  "human_5",
  "https://pixel-shot.com/en/showcase/i-3650244-0/attractive-man-in-formal-suit-on-white-background"
 ],
 [
  "human_6",
  "https://pixel-shot.com/en/showcase/i-1704012-0/handsome-young-man-isolated-on-white"
 ],
 [
  "human_7",
  "https://pixel-shot.com/en/showcase/i-3946723-0/young-man-posing-for-professional-photographer-on-white-background-in-studio"
 ],
 [
  "horse_h1",
  "https://pixel-shot.com/en/showcase/i-997934-0/beautiful-girl-riding-a-horse-outdoors"
 ],
 [
  "horse_h2",
  "https://pixel-shot.com/en/showcase/i-8475295-0/beautiful-woman-dressed-for-halloween-with-horse-outdoors"
 ],
 [
  "human_h1",
  "https://pixel-shot.com/en/showcase/i-997697-0/elegant-woman-in-costume-for--horseback-riding-outdoors"
 ]
]
fetch_images(HH_TEST, 'test_images')
show_folder('test_images', 'Тестові зображення: коні та люди')

## Допоміжні функції (код моделі — з Завдання 1)

In [ ]:
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if logs.get('accuracy') > 0.999:
            print("\nДосягнуто 99,9% точності, тому скасовуємо навчання!")
            self.model.stop_training = True

AUG = dict(rotation_range=40, width_shift_range=0.2, height_shift_range=0.2,
           shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest')

def make_generators(size, augment=False):
    train_datagen = ImageDataGenerator(rescale=1./255, **(AUG if augment else {}))
    validation_datagen = ImageDataGenerator(rescale=1./255)      # перевірочні дані не доповнюємо
    tg = train_datagen.flow_from_directory(train_dir, batch_size=32, class_mode='binary', target_size=(size, size))
    vg = validation_datagen.flow_from_directory(validation_dir, batch_size=32, class_mode='binary', target_size=(size, size))
    return tg, vg

def build_model(size):
    base = InceptionV3(input_shape=(size, size, 3), include_top=False, weights=None)
    base.load_weights('/tmp/inception_v3_weights.h5')
    for layer in base.layers:
        layer.trainable = False
    last = base.get_layer('mixed7')
    x = layers.Flatten()(last.output)
    x = layers.Dense(1024, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(1, activation='sigmoid')(x)
    model = Model(inputs=base.input, outputs=x)
    model.compile(optimizer=RMSprop(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    print(f'{size}×{size}: вихід mixed7 {last.output_shape}')
    return model, base

def n_trainable(model):
    return int(sum(np.prod(w.shape) for w in model.trainable_weights))

def predict_test(model, size, title):
    fs = sorted(f for f in os.listdir('test_images') if f.endswith('.jpg'))
    rows, cols = [], 6
    plt.figure(figsize=(cols * 2.6, int(np.ceil(len(fs) / cols)) * 2.9))
    for i, f in enumerate(fs):
        x = img_to_array(load_img(os.path.join('test_images', f), target_size=(size, size))) / 255.
        prob = float(model.predict(x[None], verbose=0)[0][0])
        ans = 'людина' if prob > 0.5 else 'кінь'
        truth = 'кінь' if f.startswith('horse') else 'людина'
        conf = prob if prob > 0.5 else 1 - prob
        plt.subplot(int(np.ceil(len(fs) / cols)), cols, i + 1); plt.imshow(load_img(os.path.join('test_images', f))); plt.axis('off')
        plt.title(f'{ans} ({conf*100:.0f}%)\n{f}', fontsize=8, color='green' if ans == truth else 'red')
        rows.append({'файл': f, 'справжній': truth, 'відповідь': ans, 'впевненість, %': round(conf * 100, 1), 'вірно': ans == truth})
    plt.suptitle(title); plt.tight_layout(); plt.show()
    df = pd.DataFrame(rows)
    norm, hard = df[~df['файл'].str.contains('_h')], df[df['файл'].str.contains('_h')]
    print(f'{title}: звичайні — вірно {norm["вірно"].sum()} з {len(norm)}, «обманні» — {hard["вірно"].sum()} з {len(hard)}')
    return df

def plot_history(h, title):
    h = h.history; ep = range(1, len(h['loss']) + 1)
    plt.figure(figsize=(13, 4))
    plt.subplot(1, 2, 1); plt.plot(ep, h['accuracy'], 'b-o', ms=3, label='Навчальні'); plt.plot(ep, h['val_accuracy'], 'r-o', ms=3, label='Перевірочні')
    plt.title('Точність'); plt.xlabel('Епоха'); plt.legend(); plt.grid(alpha=.3)
    plt.subplot(1, 2, 2); plt.plot(ep, h['loss'], 'b-o', ms=3, label='Навчальні'); plt.plot(ep, h['val_loss'], 'r-o', ms=3, label='Перевірочні')
    plt.title('Помилка (loss)'); plt.xlabel('Епоха'); plt.legend(); plt.grid(alpha=.3)
    plt.suptitle(title); plt.tight_layout(); plt.show()

results, tests = [], {}
def experiment(key, name, size, augment=False, epochs=100, model=None):
    tg, vg = make_generators(size, augment)
    if model is None:
        model, _ = build_model(size)
    t0 = time.time()
    h = model.fit(tg, validation_data=vg, epochs=epochs, verbose=2, callbacks=[myCallback()])
    t = time.time() - t0
    hh = h.history
    plot_history(h, f'{key}. {name}')
    df = predict_test(model, size, f'{key}. {name}')
    tests[key] = df
    norm, hard = df[~df['файл'].str.contains('_h')], df[df['файл'].str.contains('_h')]
    results.append({'експеримент': f'{key}. {name}', 'усього параметрів': model.count_params(), 'навчаються': n_trainable(model),
                    'епох': len(hh['loss']), 'час, с': round(t), 'с/епоху': round(t / len(hh['loss']), 1),
                    'train acc, %': round(hh['accuracy'][-1] * 100, 2), 'val acc, %': round(hh['val_accuracy'][-1] * 100, 2),
                    'найкраща val acc, %': round(max(hh['val_accuracy']) * 100, 2), 'val loss': round(hh['val_loss'][-1], 4),
                    'нові фото': f"{norm['вірно'].sum()}/{len(norm)}", '«обманні»': f"{hard['вірно'].sum()}/{len(hard)}"})
    return model, h

## A. Перенесення навчання 150×150 без аугментації (Завдання 1) + розпізнавання нових фото

In [ ]:
model_a, hist_a = experiment('A', '150×150, без аугментації', 150)

## B. Оригінальна роздільна здатність 300×300
Карта ознак mixed7 стає 17×17×768 замість 7×7×768, тому шар Flatten → Dense 1024 має в ~6 разів більше параметрів.

In [ ]:
model_b, hist_b = experiment('B', '300×300, без аугментації', 300)
del model_b; tf.keras.backend.clear_session()

## C. Перенесення навчання + Data Augmentation
Навчальний генератор випадково повертає (до 40°), зсуває (20%), перекошує, масштабує і віддзеркалює кожне фото — щоепохи мережа бачить «нові» варіанти тих самих зображень.
Перевірочний генератор окремий, без доповнення. Навчальна точність з аугментацією росте повільніше, тому обмежуємо 15 епохами.

In [ ]:
model_c, hist_c = experiment('C', '150×150 + аугментація', 150, augment=True, epochs=15)

## D. Перенесення навчання + Fine Tuning
Беремо модель A (вже навчений класифікатор) і розморожуємо **ще один згортковий шар** — останній Conv2D перед mixed7,
донавчаємо з малою швидкістю RMSprop 1e-5.

In [ ]:
base_a = model_a  # у функціональній моделі шари InceptionV3 — частина model_a
names = [l.name for l in model_a.layers]
conv = [l for l in model_a.layers[:names.index('mixed7') + 1] if isinstance(l, layers.Conv2D)][-1]
print('Розморожуємо шар:', conv.name, conv.kernel.shape)
before = n_trainable(model_a)
conv.trainable = True
model_a.compile(optimizer=RMSprop(learning_rate=1e-5), loss='binary_crossentropy', metrics=['accuracy'])
print(f'Параметрів, що навчаються: {before:,} → {n_trainable(model_a):,}')
model_d, hist_d = experiment('D', '150×150 + fine tuning (1 згортковий шар)', 150, epochs=8, model=model_a)

## Порівняння

In [ ]:
res = pd.DataFrame(results)
res.to_csv('lab10_2_horses_humans.csv', index=False)
pd.set_option('display.max_columns', 20)
res

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for (k, h) in [('A 150', hist_a), ('B 300', hist_b), ('C аугментація', hist_c), ('D fine tuning', hist_d)]:
    ax[0].plot(range(1, len(h.history['val_accuracy']) + 1), h.history['val_accuracy'], '-o', ms=3, label=k)
    ax[1].plot(range(1, len(h.history['val_loss']) + 1), h.history['val_loss'], '-o', ms=3, label=k)
ax[0].set_title('Точність на перевірочних даних'); ax[1].set_title('Помилка на перевірочних даних')
for a in ax: a.set_xlabel('Епоха'); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.show()
tab = pd.DataFrame({k: v.set_index('файл')['відповідь'] for k, v in tests.items()})
tab.insert(0, 'справжній', tests['A'].set_index('файл')['справжній'])
tab

In [ ]:
model_c.save('lab10_2_horses_humans_aug.keras')
!zip -q -r lab10_horses_humans_test_images.zip test_images
from google.colab import files
for f in ['lab10_2_horses_humans.csv', 'lab10_2_horses_humans_aug.keras', 'lab10_horses_humans_test_images.zip']:
    files.download(f)